In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = (SparkSession.builder
         .master("spark://spark-master:7077")
        .getOrCreate()
        )


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/29 04:07:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [10]:

yellow_trip_data = "/data/archive/yellow_tripdata_2019-01.csv"

df = spark.read.option("header", True).csv(yellow_trip_data) # transformation

df.show() # action

df.printSchema()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|              1|         1.50|         1|                 N|         151|         239|           1|          7|  0.5|    0.5|      1.65|           0|                  0.3

### The StructType 

The StructType and StructField classes in PySpark are used to specify the custom schema to the DataFrame and create complex columns like nested struct, array, and map columns. StructType is a collection of StructField objects that define column name, column data type, boolean to specify if the field can be nullable or not, and metadata.

In [9]:
from pyspark.sql.types import *

df_schema = StructType([
    StructField("VendorID", StringType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", IntegerType(), True),
    StructField("trip_distance", FloatType(), True),
    StructField("RatecodeID", IntegerType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("payment_type", IntegerType(), True),
    StructField("fare_amount", FloatType(), True),
    StructField("extra", FloatType(), True),
    StructField("mta_tax", FloatType(), True),
    StructField("tip_amount", FloatType(), True),
    StructField("tolls_amount", FloatType(), True),
    StructField("improvement_surcharge", FloatType(), True),
    StructField("total_amount", FloatType(), True),
    StructField("congestion_surcharge", FloatType(), True)  
])

In [15]:
df_2 = (spark.read
            .option("header", True)
            .schema(df_schema)
            .csv(yellow_trip_data)
)

df_2.show()
df_2.printSchema()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|              1|          1.5|         1|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.65|         0.0|                  0.3

In [10]:
yellow_trip_data_all = "/data/archive/*.csv"

df_3 = (spark.read
            .option("header",True)
            .schema(df_schema)
            .csv(yellow_trip_data_all)
)

df_3.show()
df_3.count()



+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|              1|          1.5|         1|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.65|         0.0|                  0.3

101246797

In [3]:
def ingest_data(spark, source_path, output_path=None, format_type="parquet"):
    """
    Ingests data following best practices

    Args:
        spark: SparkSession
        source_path: Path to source file
        output_path: Where to save processed data (Optional)
        format_type: Output format (default: parquet)
    """

    print(f"Ingesting data from: {source_path}")
    source_type = source_path.split(".")[-1].lower()

    if source_type in ("json", "jsonl"):
        df = spark.read.json(sorce_path)
    elif source_type == "csv":
        df = (spark.read
                 .option("header", True)
                 .option("inferSchema", True)
                 .csv(source_path)
             )
    elif source_type == "parquet":
        df = spark.read.parquet(source_path)
    else:
        print(f"Unsupported format: {source_type}")
        return None

    record_count = df.count()
    column_count = len(df.columns)
    print(f"Loaded {record_count} records with {column_count} columns")

    if output_path:
        print(f"Saving to {output_path} in {format_type} format")
        if format_type == "parquet":
            df.write.mode("overwrite").parquet(output_path)
        elif format_type == "csv":
            df.write.mode("overwrite").option("header", True).csv(output_path)
        else:
            df.write.mode("overwrite").format(format_type).save(output_path)
            
    return df

In [5]:
yellow_trip_data_all = "/data/archive/*.csv"

test = ingest_data(
        spark,
        yellow_trip_data_all,
        ("/data/output"),
        "parquet"
)

print(test)


Ingesting data from: /data/archive/*.csv


Loaded 101246797 records with 18 columns
Saving to /data/output in parquet format


DataFrame[VendorID: int, tpep_pickup_datetime: timestamp, tpep_dropoff_datetime: timestamp, passenger_count: int, trip_distance: double, RatecodeID: int, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: int, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double]


26/01/29 04:20:56 ERROR SparkContext: Exception getting heap histogram from executor driver
java.io.IOException: Cannot run program "/opt/java/openjdk/bin/jmap": error=2, No such file or directory
	at java.base/java.lang.ProcessBuilder.start(Unknown Source)
	at java.base/java.lang.ProcessBuilder.start(Unknown Source)
	at org.apache.spark.util.Utils$.getHeapHistogram(Utils.scala:2147)
	at org.apache.spark.SparkContext.getExecutorHeapHistogram(SparkContext.scala:748)
	at org.apache.spark.ui.exec.ExecutorHeapHistogramPage.render(ExecutorHeapHistogramPage.scala:41)
	at org.apache.spark.ui.WebUI.$anonfun$attachPage$1(WebUI.scala:90)
	at org.apache.spark.ui.JettyUtils$$anon$1.doGet(JettyUtils.scala:81)
	at javax.servlet.http.HttpServlet.service(HttpServlet.java:503)
	at javax.servlet.http.HttpServlet.service(HttpServlet.java:590)
	at org.sparkproject.jetty.servlet.ServletHolder.handle(ServletHolder.java:799)
	at org.sparkproject.jetty.servlet.ServletHandler$ChainEnd.doFilter(ServletHandler.j

In [11]:
df2 = spark.read.parquet("/data/output/*.parquet")
df2.count()
df2.show()

[Stage 20:>                                                         (0 + 1) / 1]

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|              1|          1.5|         1|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.65|         0.0|                  0.3